# 04 · Potencjały wywołane (ERP) i analiza czas–częstotliwość

**Kurs:** Praktyczna Analiza Danych Eye-Tracking i EEG w Pythonie

Pojedyncza odpowiedź mózgu na bodziec ginie w „szumie” trwającej aktywności EEG. Jeśli jednak powtórzymy bodziec wiele razy i **uśrednimy** fragmenty sygnału wokół jego pojawienia się, aktywność niezwiązana z bodźcem się zniesie, a zostanie **potencjał wywołany** (*Event-Related Potential*, ERP). W tym notebooku:

1. wytniemy epoki wokół bodźców wzrokowych (`mne.Epochs`),
2. policzymy i narysujemy ERP z zacienieniem **błędu standardowego** (SEM),
3. narysujemy **mapy topograficzne** (`plot_topomap`) i zobaczymy, która półkula odpowiada na bodziec,
4. wykonamy **analizę czas–częstotliwość** (TFA) falkami Morleta (`mne.time_frequency.tfr_morlet`).

**Dane:** oczyszczony sygnał EEG z notebooka 03 (zestaw `sample`). W tym badaniu w lewym lub prawym polu widzenia wyświetlano szachownice, a do lewego lub prawego ucha podawano dźwięki. Rzadko pojawiała się też „buźka”, na którą badany odpowiadał naciśnięciem przycisku.

**Szacowany czas:** ok. 90 minut.

## 0. Przygotowanie środowiska i wczytanie danych

Wczytujemy dane oczyszczone w notebooku 03. Jeśli tego pliku nie ma (np. notebook 03 nie był uruchamiany), korzystamy z gotowej wersji przefiltrowanej 0–40 Hz z zestawu `sample`. Nie usunięto z niej jednak mrugnięć, dlatego epoki z mrugnięciami odrzucimy progiem amplitudy w kanale EOG.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
from mne.time_frequency import tfr_morlet

KATALOG_GLOWNY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
WYNIKI = KATALOG_GLOWNY / "wyniki"
WYNIKI.mkdir(exist_ok=True)

mne.set_log_level("WARNING")
mne.viz.set_browser_backend("matplotlib")
KOLORY = {"lewe": "#2a78d6", "prawe": "#eb6834"}  # paleta czytelna przy zaburzeniach widzenia barw
plt.rcParams.update({"figure.figsize": (12, 4)})

plik_oczyszczony = WYNIKI / "03_sample_eeg_oczyszczone_raw.fif"
if plik_oczyszczony.exists():
    raw = mne.io.read_raw_fif(plik_oczyszczony, preload=True)
    progi_odrzucania = dict(eeg=100e-6)
    print("Wczytano dane oczyszczone w notebooku 03.")
else:
    katalog_sample = mne.datasets.sample.data_path()
    raw = mne.io.read_raw_fif(katalog_sample / "MEG" / "sample" / "sample_audvis_filt-0-40_raw.fif", preload=True)
    raw.pick(["eeg", "eog", "stim"]).del_proj()
    raw.interpolate_bads(reset_bads=True)
    raw.filter(l_freq=1.0, h_freq=None, picks=["eeg", "eog"])
    raw.set_eeg_reference("average", projection=False)
    progi_odrzucania = dict(eeg=100e-6, eog=200e-6)  # odrzucamy epoki z mrugnięciami
    print("Brak wyników notebooka 03 – używam wersji 0–40 Hz z zestawu sample (bez ICA).")
raw

## 1. Zdarzenia (*events*)

Momenty pojawienia się bodźców zapisano w kanale wyzwalaczy `STI 014` jako kody liczbowe. Funkcja `mne.find_events` zwraca tablicę o trzech kolumnach: **numer próbki**, wartość poprzednia (zwykle 0) i **kod zdarzenia**. Nadajemy kodom czytelne nazwy – ukośnik tworzy **hierarchię** nazw, dzięki czemu `epoki["wzrokowe"]` wybierze jednocześnie bodźce lewe i prawe.

In [ ]:
zdarzenia = mne.find_events(raw, stim_channel="STI 014")
kody_zdarzen = {"sluchowe/lewe": 1, "sluchowe/prawe": 2, "wzrokowe/lewe": 3, "wzrokowe/prawe": 4,
                "buzka": 5, "przycisk": 32}

print("Pierwsze zdarzenia (próbka, poprzednia wartość, kod):")
print(zdarzenia[:5])
for nazwa, kod in kody_zdarzen.items():
    print(f"{nazwa:15s} (kod {kod:2d}): {np.sum(zdarzenia[:, 2] == kod):3d} zdarzeń")

In [ ]:
fig = mne.viz.plot_events(zdarzenia, sfreq=raw.info["sfreq"], first_samp=raw.first_samp,
                          event_id=kody_zdarzen, show=False)
fig.axes[0].set_title("Zdarzenia w czasie nagrania")
plt.show()

odstepy = np.diff(zdarzenia[:, 0]) / raw.info["sfreq"]
print(f"Mediana odstępu między kolejnymi bodźcami: {np.median(odstepy):.2f} s")

## 2. Epokowanie wokół bodźców wzrokowych

Obiekt `mne.Epochs` wycina z ciągłego sygnału fragmenty wokół wybranych zdarzeń:

* `tmin=-0.2, tmax=0.5` – od 200 ms przed do 500 ms po bodźcu,
* `baseline=(None, 0)` – **korekcja linii bazowej**: od każdej epoki odejmujemy średnią z okresu przed bodźcem, aby usunąć przesunięcia niezwiązane z bodźcem,
* `reject` – odrzucamy epoki, w których amplituda międzyszczytowa przekracza próg (pozostałości artefaktów).

In [ ]:
kody_wzrokowe = {"wzrokowe/lewe": 3, "wzrokowe/prawe": 4}
epoki = mne.Epochs(raw, zdarzenia, event_id=kody_wzrokowe, tmin=-0.2, tmax=0.5, baseline=(None, 0),
                   reject=progi_odrzucania, picks=["eeg", "eog"], preload=True)
epoki.pick("eeg")  # kanał EOG był potrzebny tylko do ewentualnego odrzucania epok z mrugnięciami
print(epoki)
print(f"Odrzucono {epoki.drop_log_stats():.1f}% epok")

Do dalszych analiz wybieramy trzy elektrody potyliczne: parę symetryczną nad lewą i prawą półkulą oraz elektrodę przy linii środkowej. Zestaw `sample` nie używa nazw 10–20, więc wybieramy je na podstawie współrzędnych (x > 0 to prawa strona głowy, y < 0 – tył głowy).

In [ ]:
LEWY, SRODKOWY, PRAWY = "EEG 054", "EEG 058", "EEG 056"
for kanal in (LEWY, SRODKOWY, PRAWY):
    x, y, z = epoki.info["chs"][epoki.ch_names.index(kanal)]["loc"][:3] * 100
    print(f"{kanal}: x = {x:5.1f} cm, y = {y:5.1f} cm, z = {z:5.1f} cm")

Zanim uśrednimy, spójrzmy na **pojedyncze próby**. Obraz epok (*ERP image*) pokazuje każdą epokę jako jeden wiersz – widać, że odpowiedź jest powtarzalna, ale w pojedynczej próbie ginie w szumie. Pod spodem – średnia ze wszystkich prób.

In [ ]:
figs = epoki["wzrokowe/lewe"].plot_image(picks=PRAWY, title=f"{PRAWY}: bodźce w lewym polu widzenia")

## 3. Potencjały wywołane (ERP)

Metoda `average()` uśrednia epoki i zwraca obiekt `mne.Evoked`. Wykres `plot_joint` łączy przebiegi wszystkich kanałów (*butterfly plot*) z mapami topograficznymi w wybranych chwilach.

In [ ]:
evoked_lewe = epoki["wzrokowe/lewe"].average()
evoked_prawe = epoki["wzrokowe/prawe"].average()
evoked_wzrokowe = epoki["wzrokowe"].average()

fig = evoked_wzrokowe.plot_joint(times=[0.09, 0.17, 0.26], title="ERP – wszystkie bodźce wzrokowe")

Widać typową sekwencję wzrokowych komponentów ERP: dodatni **P1** (ok. 90–110 ms) nad korą potyliczną, ujemny **N1** (ok. 150–180 ms) i późniejszą dodatnią falę.

### ERP z zacienieniem błędu standardowego

Sama średnia nie mówi, jak bardzo jesteśmy jej pewni. Zacieniony pas **± 1 SEM** (błąd standardowy średniej, $\mathrm{SEM} = s / \sqrt{n}$) pokazuje zmienność między próbami. Najpierw policzymy go „ręcznie” w NumPy.

In [ ]:
def srednia_i_sem(epoki, kanal):
    """Średnia i błąd standardowy średniej dla jednego kanału (w µV)."""
    dane = epoki.get_data(picks=kanal)[:, 0, :] * 1e6  # (epoki × czas), V → µV
    return dane.mean(axis=0), dane.std(axis=0, ddof=1) / np.sqrt(len(dane))


czas_ms = epoki.times * 1000
fig, osie = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for os_, kanal, polkula in zip(osie, [LEWY, PRAWY], ["lewa półkula", "prawa półkula"]):
    for strona, opis in [("lewe", "lewym"), ("prawe", "prawym")]:
        srednia, sem = srednia_i_sem(epoki[f"wzrokowe/{strona}"], kanal)
        os_.plot(czas_ms, srednia, color=KOLORY[strona], lw=2, label=f"bodziec w {opis} polu widzenia")
        os_.fill_between(czas_ms, srednia - sem, srednia + sem, color=KOLORY[strona], alpha=0.25, lw=0)
    os_.axvline(0, color="black", lw=1)
    os_.axhline(0, color="black", lw=0.5)
    os_.set(title=f"{kanal} ({polkula})", xlabel="Czas od bodźca [ms]", ylabel="Amplituda [µV]")
    os_.grid(alpha=0.3)
osie[0].legend(loc="lower left")
fig.suptitle("ERP (średnia ± SEM) nad lewą i prawą korą potyliczną")
plt.tight_layout()
plt.show()

Komponenty P1 i N1 są wyraźnie silniejsze nad półkulą **przeciwną** (kontralateralną) do pola widzenia, w którym pojawił się bodziec. To bezpośrednie odbicie anatomii drogi wzrokowej: lewe pole widzenia jest reprezentowane w prawej korze wzrokowej i odwrotnie.

Ten sam wykres można uzyskać funkcją `mne.viz.plot_compare_evokeds`. Przekazujemy jej listy **pojedynczych prób** (`iter_evoked()`), a parametrem `ci` – funkcję liczącą granice pasma (tu: średnia ± SEM).

In [ ]:
def pasmo_sem(dane):
    """Granice pasma średnia ± SEM; dane: tablica (obserwacje × czas)."""
    srednia = dane.mean(axis=0)
    sem = dane.std(axis=0, ddof=1) / np.sqrt(len(dane))
    return np.array([srednia - sem, srednia + sem])


proby = {"lewe pole widzenia": list(epoki["wzrokowe/lewe"].iter_evoked()),
         "prawe pole widzenia": list(epoki["wzrokowe/prawe"].iter_evoked())}
figs = mne.viz.plot_compare_evokeds(proby, picks=PRAWY, ci=pasmo_sem,
                                    colors={"lewe pole widzenia": KOLORY["lewe"],
                                            "prawe pole widzenia": KOLORY["prawe"]},
                                    title=f"{PRAWY} (prawa półkula): średnia ± SEM")

## 4. Mapy topograficzne

Mapa topograficzna pokazuje rozkład potencjału na powierzchni głowy w danej chwili (wartości między elektrodami są interpolowane). Paleta jest **dywergentna**: czerwień – potencjał dodatni, błękit – ujemny, biel – zero. Ustalamy wspólną skalę (`vlim`), aby mapy dało się porównywać.

In [ ]:
czasy_map = [0.09, 0.12, 0.17, 0.25]
for evoked, opis in [(evoked_lewe, "lewym"), (evoked_prawe, "prawym")]:
    fig = evoked.plot_topomap(times=czasy_map, ch_type="eeg", vlim=(-10, 10), show=False)
    fig.suptitle(f"Bodziec w {opis} polu widzenia", fontweight="bold")
    plt.show()

Lateralizację widać jeszcze lepiej na **różnicy** odpowiedzi (lewe − prawe pole widzenia), którą liczy `mne.combine_evoked`.

In [ ]:
roznica = mne.combine_evoked([evoked_lewe, evoked_prawe], weights=[1, -1])
fig = roznica.plot_topomap(times=[0.09, 0.17], ch_type="eeg", show=False)
fig.suptitle("Różnica: lewe − prawe pole widzenia", fontweight="bold")
plt.show()

Funkcja niższego poziomu `mne.viz.plot_topomap` rysuje mapę dla **dowolnego wektora wartości** (po jednej na kanał). Narysujmy średnią amplitudę w oknie komponentu P1 (80–120 ms) – tak mierzy się amplitudę komponentów w wielu publikacjach.

In [ ]:
okno_p1 = (0.08, 0.12)
fig, osie = plt.subplots(1, 2, figsize=(9, 4))
for os_, (evoked, opis) in zip(osie, [(evoked_lewe, "lewe pole widzenia"), (evoked_prawe, "prawe pole widzenia")]):
    srednia_p1 = evoked.copy().crop(*okno_p1).data.mean(axis=1) * 1e6  # µV, jedna wartość na kanał
    obraz_mapy, _ = mne.viz.plot_topomap(srednia_p1, evoked.info, axes=os_, vlim=(-8, 8), cmap="RdBu_r", show=False)
    os_.set_title(opis)
fig.colorbar(obraz_mapy, ax=osie, label="Średnia amplituda 80–120 ms [µV]", shrink=0.8)
plt.show()

## 5. Analiza czas–częstotliwość (TFA)

ERP pokazuje tylko aktywność **zsynchronizowaną w fazie** z bodźcem – przy uśrednianiu wszystko inne się znosi. Bodziec zmienia jednak także **moc oscylacji** (np. spadek rytmu alfa), nawet jeśli faza fal jest w każdej próbie inna. Taką aktywność widać w analizie czas–częstotliwość, w której moc liczymy **w każdej próbie osobno**, a dopiero potem uśredniamy.

### Falki Morleta

Falka Morleta to sinusoida „przycięta” obwiednią Gaussa. Splot sygnału z falką o częstotliwości *f* mówi, ile jest aktywności o tej częstotliwości w danej chwili. Parametr `n_cycles` (liczba cykli w falce) ustala kompromis: więcej cykli to lepsza rozdzielczość w częstotliwości, ale gorsza w czasie.

In [ ]:
fs = raw.info["sfreq"]
fig, ax = plt.subplots(figsize=(10, 3.5))
for f, kolor in [(6, "#2a78d6"), (10, "#eb6834"), (20, "#1baf7a")]:
    falka = mne.time_frequency.morlet(fs, freqs=[f], n_cycles=f / 2)[0]
    t = (np.arange(len(falka)) - len(falka) // 2) / fs
    ax.plot(t * 1000, falka.real / np.abs(falka).max(), color=kolor, label=f"{f} Hz, n_cycles = {f / 2:g}")
ax.set(xlabel="Czas [ms]", ylabel="Część rzeczywista (znormalizowana)",
       title="Falki Morleta przy n_cycles = f / 2 (stała długość ok. 0,5 s)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

Do TFA potrzebujemy **dłuższych epok** niż do ERP – falki dla niskich częstotliwości są długie, a na brzegach epoki wynik jest zniekształcony (efekty brzegowe). Wycinamy więc epoki od −0,5 do 1,0 s, a wyniki pokażemy tylko dla przedziału −0,3…0,7 s.

Funkcja `tfr_morlet` zwraca:

* **moc** (*power*) – średnią z prób moc w każdym punkcie czas × częstotliwość,
* **ITC** (*Inter-Trial Coherence*) – spójność fazy między próbami (0 = losowa faza, 1 = identyczna faza w każdej próbie).

Obliczenia mogą zająć kilkadziesiąt sekund.

In [ ]:
epoki_tfa = mne.Epochs(raw, zdarzenia, event_id=kody_wzrokowe, tmin=-0.5, tmax=1.0, baseline=None,
                       reject=progi_odrzucania, picks=["eeg", "eog"], preload=True).pick("eeg")

czestotliwosci = np.arange(4, 41, 2)  # 4, 6, …, 40 Hz
n_cykli = czestotliwosci / 2  # stała długość falki ≈ 0,5 s

moc, itc = tfr_morlet(epoki_tfa, freqs=czestotliwosci, n_cycles=n_cykli, return_itc=True, decim=3, average=True)
# Nowszy, równoważny zapis w MNE:
# moc, itc = epoki_tfa.compute_tfr("morlet", freqs=czestotliwosci, n_cycles=n_cykli,
#                                  return_itc=True, decim=3, average=True)
moc

Moc wyrażamy względem **linii bazowej** sprzed bodźca (tryb `logratio`: logarytm ilorazu mocy do mocy w linii bazowej). Wartości dodatnie (czerwone) oznaczają wzrost mocy po bodźcu, ujemne (niebieskie) – spadek.

In [ ]:
LINIA_BAZOWA_TFA = (-0.4, -0.1)
figs = moc.plot(picks=[PRAWY], baseline=LINIA_BAZOWA_TFA, mode="logratio", tmin=-0.3, tmax=0.7,
                title=f"Moc względem linii bazowej – {PRAWY}")
figs = itc.plot(picks=[PRAWY], tmin=-0.3, tmax=0.7, vlim=(0, None), cmap="Reds",
                title=f"Spójność fazy między próbami (ITC) – {PRAWY}")

Interpretacja:

* Krótko po bodźcu (ok. 50–300 ms) rośnie moc w zakresie **theta/alfa (4–12 Hz)**, a ITC jest wysokie – to w dużej mierze ten sam proces, który widzimy jako ERP (odpowiedź zsynchronizowana w fazie).
* Później (ok. 300–700 ms) moc w paśmie **alfa (ok. 8–13 Hz)**, a słabiej także beta, spada poniżej linii bazowej. To desynchronizacja związana ze zdarzeniem (*Event-Related Desynchronization*, ERD), typowa dla przetwarzania wzrokowego. ITC w tym okresie jest niskie, więc tej zmiany **nie widać** w ERP.
* Uwaga: kolejne bodźce pojawiały się średnio co ok. 0,7 s, więc pod koniec epoki zaczyna się już odpowiedź na następny bodziec.

Rozkład przestrzenny tych zmian pokażą mapy topograficzne mocy w wybranych pasmach i oknach czasowych.

In [ ]:
fig, osie = plt.subplots(1, 2, figsize=(9, 4))
for os_, (t0, t1, f0, f1) in zip(osie, [(0.05, 0.25, 4, 8), (0.35, 0.65, 8, 13)]):
    moc.plot_topomap(tmin=t0, tmax=t1, fmin=f0, fmax=f1, baseline=LINIA_BAZOWA_TFA, mode="logratio",
                     cbar_fmt="%.2f", units="log₁₀(moc / linia bazowa)", axes=os_, show=False)
osie[0].set_title("Theta 4–8 Hz, 50–250 ms")
osie[1].set_title("Alfa 8–13 Hz, 350–650 ms")
plt.show()

## 6. Zapis wyników

In [ ]:
mne.write_evokeds(WYNIKI / "04_sample_erp_wzrokowe-ave.fif", [evoked_lewe, evoked_prawe], overwrite=True)
print("Zapisano:", *sorted(p.name for p in WYNIKI.glob("04_*")), sep="\n  ")

## Podsumowanie

* `mne.Epochs` wycina fragmenty sygnału wokół zdarzeń; korekcja linii bazowej i progi odrzucania to standardowe elementy tego kroku.
* Uśrednienie epok daje ERP (`mne.Evoked`); pas **± SEM** pokazuje niepewność średniej.
* Mapy topograficzne ujawniają **lateralizację** wczesnych komponentów wzrokowych (P1, N1) – odpowiedź jest silniejsza nad półkulą przeciwną do pola widzenia.
* Analiza czas–częstotliwość falkami Morleta pokazuje także zmiany mocy **niezsynchronizowane w fazie** z bodźcem (np. spadek alfa), których ERP nie widzi.

## Zadania do samodzielnego wykonania

Pięć zadań, w których powtórzysz najważniejsze kroki z tego notebooka. **Nie wymagają nowych funkcji ani pojęć** – wszystko, czego potrzebujesz, pojawiło się wyżej (przy każdym zadaniu podajemy, którą sekcję powtarza). Każde zadanie kończy się **jednym wynikiem**: liczbą wypisaną w podanym formacie albo jedną figurą.

**Zasady pracy**

1. Zanim zaczniesz, uruchom cały notebook (**Run All**) – zadania korzystają ze zmiennych i funkcji zdefiniowanych wyżej.
2. Rozwiązanie wpisz w komórce kodu pod treścią zadania.
3. **Nie nadpisuj zmiennych z notebooka** (np. `raw`, `epoki`, `moc`). Nowym zmiennym nadawaj nazwy z przedrostkiem zadania: `z1_`, `z2_`, … Dzięki temu zadania nie psują sobie nawzajem danych i nie trzeba uruchamiać notebooka od nowa.
4. **Kontrola pośrednia** to wartość, którą możesz sprawdzić po drodze. Jeśli się nie zgadza, błąd jest wcześniej – popraw go, zanim pójdziesz dalej.
5. **✅ Wynik** to jedyna rzecz do oddania: liczba wypisana dokładnie w podanym formacie (z podaną liczbą miejsc po przecinku) albo figura z podanym tytułem.
6. **Do przemyślenia** – pytanie, które pomaga zrozumieć wynik. Nie trzeba go oddawać, ale warto na nie odpowiedzieć choćby w myślach.

### Zadanie 1. Próg odrzucania epok

*Powtarzasz: sekcja 2 (epokowanie, korekcja linii bazowej, odrzucanie epok).*

**Cel:** zobaczyć, jak próg amplitudy międzyszczytowej decyduje o tym, ile danych zostaje do analizy.

**Krok po kroku:**

1. Utwórz epoki bodźców wzrokowych tak jak w sekcji 2 – te same `zdarzenia`, `event_id=kody_wzrokowe`, `tmin`, `tmax` i `baseline` – ale z `picks=["eeg"]` i własnym progiem odrzucania `reject=dict(eeg=...)`.
2. Zrób to w pętli dla trzech progów: 100 µV, 50 µV i 40 µV (pamiętaj, że MNE pracuje w woltach: 40 µV = `40e-6`).
3. Dla każdego progu wypisz liczbę zachowanych epok (`len(...)`) i procent odrzuconych (`drop_log_stats()`).

**Kontrola pośrednia:** przy progu 100 µV zostają wszystkie 144 epoki – jeśli korzystasz z danych oczyszczonych w notebooku 03 (komunikat w sekcji 0).

**✅ Wynik:** `print(f"Zadanie 1: przy progu 40 µV zostało {z1_n_40} epok")`

**Do przemyślenia:** Co zyskujemy, a co tracimy, zaostrzając próg? Jak zmieni się szerokość pasma SEM, gdy epok będzie mniej? (Możesz to sprawdzić funkcją `srednia_i_sem` z sekcji 3.)

In [ ]:
# Zadanie 1 – Twoje rozwiązanie



# print(f"Zadanie 1: przy progu 40 µV zostało {z1_n_40} epok")

### Zadanie 2. Lateralizacja P1 w liczbach

*Powtarzasz: sekcje 3–4 (ERP dla lewego i prawego pola widzenia, średnia amplituda w oknie).*

**Cel:** zmierzyć lateralizację komponentu P1 tak, jak robi się to w publikacjach – średnią amplitudą w oknie czasowym.

**Krok po kroku:**

1. Użyj gotowych średnich `evoked_lewe` i `evoked_prawe` (sekcja 3).
2. Dla kanałów `LEWY` (EEG 054, lewa półkula) i `PRAWY` (EEG 056, prawa półkula) policz **średnią amplitudę w oknie P1 (80–120 ms)** w µV – tak jak w ostatniej komórce sekcji 4: `evoked.copy().crop(0.08, 0.12)`, potem `get_data(picks=kanal).mean() * 1e6`.
3. Wypisz wszystkie cztery wartości (2 kanały × 2 pola widzenia).
4. Dla każdego kanału policz różnicę: bodziec **przeciwstronny** (kontralateralny) minus **tożstronny** (ipsilateralny). Uwaga: dla kanału nad **lewą** półkulą przeciwstronne jest **prawe** pole widzenia.

**Kontrola pośrednia:** dla kanału `PRAWY` różnica (lewe pole − prawe pole) wynosi ok. 1,1 µV.

**✅ Wynik:** `print(f"Zadanie 2: P1 w EEG 054 (kontra − ipsi) = {z2_lewy:.1f} µV")`

**Do przemyślenia:** Nad lewą półkulą lateralizacja P1 jest dużo silniejsza niż nad prawą. Spójrz na wykres ERP z sekcji 3: który komponent silniej lateralizuje się nad prawą półkulą?

In [ ]:
# Zadanie 2 – Twoje rozwiązanie



# print(f"Zadanie 2: P1 w EEG 054 (kontra − ipsi) = {z2_lewy:.1f} µV")

### Zadanie 3. Topografia odpowiedzi słuchowej

*Powtarzasz: sekcje 2 i 4 (epokowanie, mapa topograficzna średniej amplitudy w oknie).*

**Cel:** przećwiczyć całą drogę od zdarzeń do mapy topograficznej na innym typie bodźców z tego samego badania i porównać rozkład odpowiedzi słuchowej i wzrokowej.

**Krok po kroku:**

1. Utwórz epoki dla bodźców **słuchowych** (`event_id={"sluchowe/lewe": 1, "sluchowe/prawe": 2}`) z tymi samymi parametrami co epoki wzrokowe w sekcji 2 – łącznie z `reject=progi_odrzucania` i `picks=["eeg", "eog"]`, a potem `.pick("eeg")`.
2. Uśrednij wszystkie epoki słuchowe (`average()`).
3. Dla odpowiedzi słuchowej i dla wzrokowej (`evoked_wzrokowe` z sekcji 3) policz średnią amplitudę każdego kanału w oknie **80–120 ms** – tak jak w ostatniej komórce sekcji 4 (`crop`, `.data.mean(axis=1) * 1e6`).
4. Narysuj obie mapy obok siebie funkcją `mne.viz.plot_topomap` (`fig, osie = plt.subplots(1, 2, figsize=(9, 4))`) ze wspólną skalą `vlim=(-6, 6)` i paletą `cmap="RdBu_r"`. Dodaj pasek kolorów (`fig.colorbar`, jak w sekcji 4).
5. Tytuły paneli: „Słuchowe 80–120 ms” i „Wzrokowe 80–120 ms”; tytuł figury zacznij od „Zadanie 3”.

**Kontrola pośrednia:** epok słuchowych jest 145 (jeśli korzystasz z danych z notebooka 03).

**✅ Wynik:** jedna figura z dwiema mapami topograficznymi.

**Do przemyślenia:** Gdzie leży ujemne maksimum odpowiedzi słuchowej (komponent N1), a gdzie dodatnie maksimum odpowiedzi wzrokowej (P1)? Dlaczego w tym samym oknie czasowym rozkłady są tak różne?

In [ ]:
# Zadanie 3 – Twoje rozwiązanie



# Wynik: jedna figura z tytułem zaczynającym się od „Zadanie 3”

### Zadanie 4. Liczba cykli falki: czas czy częstotliwość?

*Powtarzasz: sekcja 5 (falki Morleta, `tfr_morlet`, moc względem linii bazowej).*

**Cel:** zobaczyć kompromis między rozdzielczością w czasie i w częstotliwości, opisany przy falkach Morleta.

**Krok po kroku:**

1. Policz moc falkami Morleta funkcją `tfr_morlet` dla epok `epoki_tfa`, dla częstotliwości **8–40 Hz co 2 Hz** (`np.arange(8, 41, 2)`) i ze **stałą** liczbą cykli: raz `n_cycles=3`, raz `n_cycles=7`. Pozostałe parametry jak w sekcji 5 (`decim=3`, `average=True`), ale `return_itc=False`. Żeby obliczenia trwały krócej, dodaj `picks=[PRAWY]`.
   > Dlaczego od 8 Hz, a nie od 4 Hz? Przy 7 cyklach falka dla 4 Hz jest dłuższa niż cała epoka (1,5 s) i MNE zgłosi błąd. Możesz to sprawdzić.
2. Narysuj obie mapy czas–częstotliwość obok siebie: `fig, osie = plt.subplots(1, 2, figsize=(13, 4))`, a w wywołaniu `.plot(...)` dodaj `axes=osie[0]` (albo `osie[1]`) i `show=False`. Użyj tej samej linii bazowej (`LINIA_BAZOWA_TFA`), trybu (`"logratio"`) i zakresu czasu (−0,3…0,7 s) co w sekcji 5 oraz wspólnej skali `vlim=(-0.3, 0.3)`.
3. Tytuły paneli: `"n_cycles = 3"` i `"n_cycles = 7"`; tytuł figury zacznij od „Zadanie 4”.

**✅ Wynik:** jedna figura z dwiema mapami czas–częstotliwość (`n_cycles` = 3 i 7).

**Do przemyślenia:** Na której mapie łatwiej określić, **kiedy** zaczyna się spadek mocy alfa, a na której – w **jakim paśmie** dokładnie występuje? Ile trwa falka 10 Hz przy 3 i przy 7 cyklach? (Sprawdź funkcją `mne.time_frequency.morlet`, jak na początku sekcji 5.)

In [ ]:
# Zadanie 4 – Twoje rozwiązanie



# Wynik: jedna figura z tytułem zaczynającym się od „Zadanie 4”

### Zadanie 5. Spadek mocy alfa w liczbach

*Powtarzasz: sekcja 5 (moc względem linii bazowej w trybie `logratio`, desynchronizacja alfa).*

**Cel:** samodzielnie policzyć to, co mapa czas–częstotliwość pokazuje kolorem, i wyrazić spadek mocy alfa jedną liczbą.

**Krok po kroku:**

1. Pobierz moc dla kanału `PRAWY` z obiektu `moc` (sekcja 5): `moc.get_data(picks=[PRAWY])[0]` – to tablica *częstotliwości × czas*; jej osie opisują `moc.freqs` i `moc.times`.
2. Policz moc w linii bazowej: dla każdej częstotliwości średnia po czasie w przedziale `LINIA_BAZOWA_TFA` (−0,4…−0,1 s). *Wskazówka:* maska `(moc.times >= -0.4) & (moc.times <= -0.1)` i `mean(axis=1, keepdims=True)`.
3. Przelicz moc na `logratio` – tak jak robi to `moc.plot(..., mode="logratio")`: $\log_{10}\left(\frac{\text{moc}}{\text{moc w linii bazowej}}\right)$.
4. Uśrednij wynik w paśmie alfa **8–13 Hz** i oknie **350–650 ms**. Zbuduj maskę na `moc.freqs` i maskę na `moc.times`, a potem wybierz wiersze i kolumny po kolei: `logratio[maska_f][:, maska_t]`.
5. Przelicz średnią $x$ na procentową zmianę mocy: $100 \cdot (10^{x} - 1)$.

**Kontrola pośrednia:** dla pasma theta (4–8 Hz) w oknie 50–250 ms wychodzi wartość dodatnia, ok. +0,32.

**✅ Wynik:** `print(f"Zadanie 5: logratio = {z5_x:.3f}, zmiana mocy alfa = {z5_procent:.0f}%")`

**Do przemyślenia:** Dlaczego ten spadek mocy jest widoczny w analizie czas–częstotliwość, a nie widać go w ERP? (Wróć do opisu ITC w sekcji 5.)

In [ ]:
# Zadanie 5 – Twoje rozwiązanie



# print(f"Zadanie 5: logratio = {z5_x:.3f}, zmiana mocy alfa = {z5_procent:.0f}%")

## Literatura

* Cohen, M. X. (2014). *Analyzing neural time series data: Theory and practice*. MIT Press.
* Gramfort, A., Luessi, M., Larson, E., i in. (2013). MEG and EEG data analysis with MNE-Python. *Frontiers in Neuroscience, 7*, 267.
* Luck, S. J. (2014). *An introduction to the event-related potential technique* (wyd. 2). MIT Press.
* Tallon-Baudry, C., i Bertrand, O. (1999). Oscillatory gamma activity in humans and its role in object representation. *Trends in Cognitive Sciences, 3*(4), 151–162.